# Part 01: Tokenizer Basics: How Text Becomes Numbers

> **Previous context**: This is the entry point, so we begin from the first question: why can a model not read raw text directly?
> **Goal for this part**: Understand what Token, Tokenizer, vocab, encode, decode, special tokens, padding, and attention masks mean.

Today we are solving one concrete confusion: what is the hidden mechanism behind this part of an LLM, and how can we rebuild it with small numbers before trusting a library?

## 0. Build the intuition

A Tokenizer is the translator in front of the model. It turns text into token IDs, and it can turn token IDs back into text. The important idea is simple: neural networks work with numbers, so text must first become a sequence of integer IDs.

## 1. Try simple choices

A character-level tokenizer is stable because every character can be represented, but it creates long sequences. A word-level tokenizer creates shorter sequences, but it breaks when it sees a word outside the vocabulary. Subword tokenization is the compromise: common pieces stay together, rare words are split into smaller pieces.

## 2. Hand-check the algorithm

For a sentence such as 'the cat', character tokenization gives ['t', 'h', 'e', ' ', 'c', 'a', 't'], while word tokenization gives ['the', 'cat']. The numbers are just IDs, like name tags; ID 12 is not 'larger in meaning' than ID 3.

## 3. Exercises

The exercises ask you to fill in encode, add BOS/EOS special tokens, and build padding plus an attention mask. Ask an AI for hints if you get stuck, but do not ask it to finish the blanks for you.

## How to use the code cells

Run the cells in order. The code is intentionally direct and small: each cell should expose one idea, print the key observation, and let you change a number to see what moves.

## Exercises

When a cell contains a TODO placeholder, fill it yourself and use the `assert` checks as feedback. You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

## Summary Checklist

- [ ] Token means the smallest text unit a model processes.
- [ ] Tokenizer.encode maps text to IDs; decode maps IDs back to text.
- [ ] Character-level is robust but long; word-level is short but brittle; subword tokenization balances both.

Next, continue through the code cells for the Foundation part and inspect the printed observations.


In [ ]:
# Teaching note: follow this line to see the main step.
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

text = "the cat sat on the mat"
ids = tokenizer.encode(text)
tokens = [tokenizer.decode([i]) for i in ids]

print(f"Original text: {text}")
print(f"token: {tokens}")
print(f"token ID: {ids}")

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

In [2]:
# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog",
    "i love my cat",
    "i love my dog",
    "the cat is cute",
    "the dog is happy",
    "the mat is soft",
    "the log is hard",
    "cats and dogs are friends",
]

print(f"Corpus size:{len(corpus)}Read the values printed above and connect them to the concept in this cell.")
for i, s in enumerate(corpus):
    print(f"  [{i}] {s}")

total_chars = sum(len(s) for s in corpus)
print(f"Total characters:{total_chars}")
print(f"Total words (split by spaces):{sum(len(s.split()) for s in corpus)}")

Corpus size:  [0] the cat sat on the mat
  [1] the dog sat on the log
  [2] the cat and the dog
  [3] i love my cat
  [4] i love my dog
  [5] the cat is cute
  [6] the dog is happy
  [7] the mat is soft
  [8] the log is hard
  [9] cats and dogs are friends

Total characters:Total words (split by spaces):

Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.
```
Text: t h e c a t        ↓    ↓    ↓    ↓    ↓    ↓    ↓
ASCII:  116  104  101  32  99   97   116
```

Read the values printed above and connect them to the concept in this cell.
- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

In [3]:
class CharTokenizer:
    'Read the values printed above and connect them to the concept in this cell.'
    
    def __init__(self):
        # Teaching note: follow this line to see the main step.
        # Teaching note: follow this line to see the main step.
        self.stoi = {}
        self.itos = {}
    
    def train(self, texts):
        'Vocabulary'
        all_text = "".join(texts)
        chars = sorted(list(set(all_text)))
        
        self.stoi = {ch: i for i, ch in enumerate(chars)}
        self.itos = {i: ch for i, ch in enumerate(chars)}
        
        print(f"Character-level tokenizer training finished")
        print(f"Vocabulary size: {len(self.stoi)}Read the values printed above and connect them to the concept in this cell.")
        print(f"Vocabulary items: {chars}")
    
    def encode(self, text):
        'Read the values printed above and connect them to the concept in this cell.'
        return [self.stoi[ch] for ch in text]
    
    def decode(self, ids):
        'Read the values printed above and connect them to the concept in this cell.'
        return "".join([self.itos[i] for i in ids])


In [ ]:
# Teaching note: follow this line to see the main step.
char_tokenizer = CharTokenizer()
char_tokenizer.train(corpus)

# Teaching note: follow this line to see the main step.
text = "the cat"
ids = char_tokenizer.encode(text)
recovered = char_tokenizer.decode(ids)

print(f"Encode/decode test")
print(f"Original text:     '{text}'")
print(f"Token IDs: {ids}")
print(f"Decoded back: '{recovered}'")
print(f"\nKey observation：Original text {len(text)}Read the values printed above and connect them to the concept in this cell.{len(ids)}Read the values printed above and connect them to the concept in this cell.")
print(f"Each character becomes one token, so there is no compression.")

In [5]:
class WordTokenizer:
    'Read the values printed above and connect them to the concept in this cell.'
    
    def __init__(self):
        self.stoi = {}
        self.itos = {}
    
    def train(self, texts):
        'Read the values printed above and connect them to the concept in this cell.'
        all_words = set()
        for text in texts:
            words = text.split()
            all_words.update(words)
        
        all_words = sorted(list(all_words))
        self.stoi = {w: i for i, w in enumerate(all_words)}
        self.itos = {i: w for i, w in enumerate(all_words)}
        
        print(f"Word-level tokenizer training finished")
        print(f"Vocabulary size: {len(self.stoi)}Read the values printed above and connect them to the concept in this cell.")
        print(f"Vocabulary items: {all_words}")
    
    def encode(self, text):
        'Read the values printed above and connect them to the concept in this cell.'
        return [self.stoi[w] for w in text.split()]
    
    def decode(self, ids):
        'Read the values printed above and connect them to the concept in this cell.'
        return " ".join([self.itos[i] for i in ids])


In [ ]:
# Teaching note: follow this line to see the main step.
word_tokenizer = WordTokenizer()
word_tokenizer.train(corpus)

text = "the cat sat on the mat"
ids = word_tokenizer.encode(text)
recovered = word_tokenizer.decode(ids)

print(f"Encode/decode test")
print(f"Original text:     '{text}'")
print(f"Token IDs: {ids}")
print(f"Decoded back: '{recovered}'")
print(f"\nKey observation：Original text {len(text.split())}Read the values printed above and connect them to the concept in this cell.{len(ids)}Read the values printed above and connect them to the concept in this cell.")
print(f"Read the values printed above and connect them to the concept in this cell.")

In [ ]:
# Teaching note: follow this line to see the main step.
print('OOV (Out Of Vocabulary) demo')
print(f"Current vocabulary:{list(word_tokenizer.stoi.keys())}")
print()

try:
    test_text = "the elephant sat"
    print(f"Trying to encode:{test_text}'")
    ids = word_tokenizer.encode(test_text)
    print(f"Result: {ids}")
except KeyError as e:
    words = test_text.split()
    for w in words:
        if w in word_tokenizer.stoi:
            print(f"  '{w}' → ID {word_tokenizer.stoi[w]}Vocabulary")
        else:
            print(f"  '{w}not in the vocabulary")
    print(f"Conclusion")
    print(f"Vocabulary")

## Summary
Read the values printed above and connect them to the concept in this cell.
- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- Read the values printed above and connect them to the concept in this cell.- OOV (Out Of Vocabulary) demo- Read the values printed above and connect them to the concept in this cell.
Read the values printed above and connect them to the concept in this cell.

In [ ]:
# Teaching note: follow this line to see the main step.
stoi = {"the": 0, "cat": 1, "sat": 2}
tokens = ["the", "cat", "sat"]

# Teaching note: follow this line to see the main step.
ids = 'TODO: replace this placeholder with your code'

assert not isinstance(ids, str), 'Please replace the placeholder before running the assertion.'
assert ids == [0, 1, 2], ids
print('Exercise passed: you have understood this step.')

In [ ]:
# Teaching note: follow this line to see the main step.
stoi = {"<BOS>": 0, "<EOS>": 1, "the": 2, "cat": 3}
tokens = ["the", "cat"]

# Teaching note: follow this line to see the main step.
input_ids = 'TODO: replace this placeholder with your code'

assert not isinstance(input_ids, str), 'Please replace the placeholder before running the assertion.'
assert input_ids == [0, 2, 3, 1], input_ids
print('Exercise passed: you have understood this step.')

In [ ]:
# Teaching note: follow this line to see the main step.
PAD_ID = 0
ids = [5, 6, 7]
max_len = 5

# Teaching note: follow this line to see the main step.
padded_ids = 'TODO: replace this placeholder with your code'
attention_mask = 'TODO: replace this placeholder with your code'

assert not isinstance(padded_ids, str), 'Please replace the placeholder before running the assertion.'
assert not isinstance(attention_mask, str), 'Please replace the placeholder before running the assertion.'
assert padded_ids == [5, 6, 7, 0, 0], padded_ids
assert attention_mask == [1, 1, 1, 0, 0], attention_mask
print('Exercise passed: you have understood this step.')